# Week 1 — Uncoalesced vs coalesced SGEMM

Measures two CUDA SGEMM kernels that do **the same arithmetic** (one thread per `C[i,j]`, K FMAs from global memory) and differ only in which thread index drives the row.

- **naive:** `threadIdx.x` → row (uncoalesced C store / A load)
- **coalesced:** `threadIdx.x` → col (coalesced C store / B load)

The harness prints device properties (name, SMs, clocks, bus width, peak DRAM GB/s), times each kernel with 3 warmup + 10 `cudaEvent` runs against cuBLAS, reports GFLOP/s, **Req GB/s** (requested traffic, not DRAM bandwidth), and gates on relative error ≤ 1e-4.

**Runtime → Change runtime type → GPU** before running anything. Free Colab is usually a T4; if the next cell prints L4 or A100, stop and say so — do not paste those numbers into the README table.

In [ ]:
import shutil
import subprocess
import sys

if shutil.which("nvidia-smi") is None:
    sys.exit(
        "No GPU in this runtime (nvidia-smi not found).\n"
        "In Colab: Runtime → Change runtime type → Hardware accelerator = GPU (T4).\n"
        "Then Runtime → Run all. A CPU runtime has no nvcc and cannot build this repo."
    )

print(subprocess.check_output(["nvidia-smi"], text=True))
gpu = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
    text=True,
).strip()
print(f"Detected GPU: {gpu}")
print("Expect T4 on free Colab. If this is L4/A100, tell me before filling the README table.")

In [ ]:
%%bash
REPO_URL="https://github.com/preethamdandu/cuda-kernel-optimization.git"
DIR="cuda-kernel-optimization"
if [ -d "$DIR/.git" ]; then
  cd "$DIR" && git pull --ff-only
else
  git clone "$REPO_URL" "$DIR"
fi

In [ ]:
%%bash
set -euo pipefail
if ! command -v nvidia-smi >/dev/null || ! command -v nvcc >/dev/null; then
  echo "No GPU / no nvcc. Runtime → Change runtime type → GPU (T4), then re-run from the top." >&2
  exit 1
fi
cd cuda-kernel-optimization
ARCH=sm_$(nvidia-smi --query-gpu=compute_cap --format=csv,noheader | head -1 | tr -d '.' | tr -d ' ')
if [ -z "${ARCH#sm_}" ]; then
  echo "Could not read compute capability. Is a GPU attached?" >&2
  exit 1
fi
echo "building for $ARCH"
nvcc -O3 -arch=$ARCH -lineinfo benchmark/bench.cu src/*.cu -lcublas -o bench
echo "build ok"

In [ ]:
%%bash
cd cuda-kernel-optimization
./bench --stage naive --sizes 1024 2048 4096
./bench --stage coalesced --sizes 1024 2048 4096

In [ ]:
%%bash
echo "Optional ncu: Colab usually fails with ERR_NVGPU_PERMISSION / counters denied."
echo "That is expected. Week 3 profiling happens on a rented 4090."
echo "Expected kernel-average sectors/request on a 4090: ~16.5 naive vs ~2.5 coalesced."
echo "Per-instruction A-load contrast is 32 vs 4; C store (op_st) is also 32 vs 4."
cd cuda-kernel-optimization
ncu --metrics \
  l1tex__t_sectors_pipe_lsu_mem_global_op_ld.sum,\
l1tex__t_requests_pipe_lsu_mem_global_op_ld.sum,\
l1tex__t_sectors_pipe_lsu_mem_global_op_st.sum,\
l1tex__t_requests_pipe_lsu_mem_global_op_st.sum \
  ./bench --stage naive --sizes 1024

Paste the two tables back into chat so the README can be filled with real numbers. Do not edit the table yourself if the GPU name is not T4 — tell me.

If the ncu cell printed `ERR_NVGPU_PERMISSION`, that is expected; Week 3 on rented 4090.